# Lab 14: OLS Diagnostics — Diagnosing an Interval That Lies
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 70 min Core + 55 min AI + 40 min Extension

---

**Learning Objectives:**
- Measure the true coverage of a nominal 95% interval under two violations: heteroskedasticity, and errors shared within a firm
- Show that robust standard errors fix heteroskedasticity and not dependence
- Read every standard error statsmodels offers, HC0 to HC3 and clustered, on one regression, and explain why clustering moves one coefficient's and not another's
- Find influential rows with leverage, studentised residuals and Cook's distance, and state what a flagging rule costs
- Write your own prompt for a function that builds that table of standard errors, and check every number it gives against statsmodels

**Dataset:** simulated data with a known truth: samples whose error spread grows with x, and a panel of 40 firms with 25 employees each

**Format:** Each diagnosis shows you output that is wrong, or looks wrong, and asks what happened. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3), the influence measures in Diagnosis 3, the AI-Assisted Expansion (your prompts, the checking log and two answers), and `src/diagnostics.py`, a finished module you save and upload.

**Time:** Core (70 min) + AI-Assisted Expansion (required, 55 min) + Extension (optional, 40 min)

With setup, the README and the GitHub submission, plan on about 2 hours 25 minutes, most of it after class. The optional Extension adds about 40 minutes.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only in the AI-Assisted Expansion, where you write the prompt yourself.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Diagnosis 3:** the influence measures for a regression with three planted rows: leverage, studentised residual and Cook's distance in one table, the planted rows, and two counts. About ten lines in an empty cell.
> - **Extension (optional):** a wild cluster bootstrap, from a three-step outline.
>
> **You write, in text cells:**
> - two answers in each of Diagnoses 1, 2 and 3;
> - **the AI-Assisted Expansion (required):** your own prompt, from scratch, asking AI for a function that builds Diagnosis 2's table of standard errors; a revised prompt for round 2 if round 1 shows a `NO`, a `missing` or an error (and round 3 if you need it); a short checking log; and two answers. Your prompts, the log and the answers are what is graded, not the AI's code, which you paste rather than type.
>
> **You run and read:** everything else. That is the two coverage simulations, the firm panel, the analyst's file and its table of standard errors, the three planted rows, and Ship It's finished module, `diagnostics.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints. The check cells in the AI section: run them; you need not follow their code.

## Setup

In [ ]:
# GIVEN — run as-is
# Setup: the libraries this lab uses
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

Path("src").mkdir(exist_ok=True)     # Ship It writes its module into this folder

RANDOM_STATE = 42                    # the seed of Diagnosis 2's firm panel
print("Setup complete.")

## Diagnosis 1: The Interval That Under-Covers (18 min)

An analyst reports a coefficient with a tight interval. The specification is
correct, the coefficient is unbiased, and the interval is wrong.

To see how wrong, you need many samples from a population whose truth you know, and a count of
how often the interval contains that truth: its **coverage**. A 95% interval should contain it in
95% of samples.

**Reading the code.** `np.random.default_rng(seed)` makes a random-number generator; the same
seed gives the same draws. `r.uniform(1, 10, n)` draws n numbers spread evenly between 1 and 10,
and `r.normal(0, 0.5 * x, n)` draws n normal errors whose standard deviation is 0.5 times each
row's x, so the spread grows with x: heteroskedasticity. `sm.add_constant(x)` adds a column of
ones, the intercept, and `sm.OLS(y, X).fit()` fits the regression, outcome first.

In [ ]:
# GIVEN — run as-is
# Diagnosis 1: one sample of n rows in which the error's spread grows with x
TRUE_SLOPE = 2.0


def one_trial(seed, n=400):
    """Draw one sample and fit OLS of y on x; the true slope is TRUE_SLOPE."""
    r = np.random.default_rng(seed)
    x = r.uniform(1, 10, n)
    y = 5 + TRUE_SLOPE * x + r.normal(0, 0.5 * x, n)     # variance grows with x
    return sm.OLS(y, sm.add_constant(x)).fit()

**The coverage count.** The next cell draws 500 samples, each with its own seed, and builds three
95% intervals for the slope in each. `fit.get_robustcov_results(cov_type="HC1")` gives back the
same fit with HC1 standard errors, and so HC1 intervals; the estimate does not change.
`.conf_int()` gives one row per coefficient, its lower and upper limit, and row 1 is the slope.
`lo <= TRUE_SLOPE <= hi` is True when the interval contains the truth, and adding True to a count
adds 1. `hits` is a dictionary, one count per name, read and updated as `hits[name]`.

The **MC SE** column is the Monte Carlo standard error of each coverage share, √(p(1 − p) / 500):
how far the share could move by chance with 500 samples. `{p:>11.1%}` prints p as a percentage
with one decimal, right-aligned in 11 characters.

In [ ]:
# GIVEN — run as-is
# Diagnosis 1, continued: in 500 samples, how often does each 95% interval contain the truth?
TRIALS = 500
hits = {"classical": 0, "HC1": 0, "HC3": 0}      # intervals that contained the true slope
for i in range(TRIALS):
    fit = one_trial(2000 + i)
    for name in hits:
        result = fit if name == "classical" else fit.get_robustcov_results(cov_type=name)
        lo, hi = result.conf_int()[1]            # row 1: the slope's interval
        hits[name] += lo <= TRUE_SLOPE <= hi     # True counts as 1

print(f"{'cov type':>12}{'coverage':>11}{'MC SE':>9}")
print("-" * 32)
for name, count in hits.items():
    p = count / TRIALS
    mc_se = (p * (1 - p) / TRIALS) ** 0.5
    print(f"{name:>12}{p:>11.1%}{mc_se:>9.1%}")

**Expected output:** against a nominal 95%, coverage of 92.4% with classical standard errors,
94.2% with HC1 and 94.4% with HC3, and Monte Carlo SEs of 1.2%, 1.0% and 1.0%.

### YOUR DIAGNOSIS

1. The estimate is unbiased throughout. Explain precisely what is broken and why it is the interval rather than the point estimate.
2. Is classical's shortfall from 95% more than two Monte Carlo SEs? Is the HC1-HC3 gap? Then say which of the three intervals head to 95% as n grows, which does not, and why.

*Your diagnosis here:*

---

## Diagnosis 2: The Violation Robust Errors Do Not Fix (25 min)

Many analysts switch on robust standard errors by habit, as if they fixed every problem with the
errors. This diagnosis tests that habit on errors that are shared within a firm.

The next cell builds a panel of 40 firms with 25 employees each. Everyone in a firm shares one
shock to the outcome, and x has a part shared within the firm as well, as training budgets and
wages would.

**Reading the code.** `np.repeat(np.arange(40), 25)` is 0 repeated 25 times, then 1 repeated 25
times, and so on up to 39: each row's firm. `r.normal(0, 1.2, N_FIRMS)` draws one shock per firm,
and `[firm]` copies each firm's draw to all 25 of its rows. `pd.DataFrame({...})` builds a table
from named columns.

In [ ]:
# GIVEN — run as-is
# Diagnosis 2: 40 firms, 25 employees each, error shared within firm
N_FIRMS, PER = 40, 25
TRUE_B = 1.5


def clustered_panel(seed):
    """One panel: each row's firm, x, and y = 1.5x + a firm shock + an individual error."""
    r = np.random.default_rng(seed)
    firm = np.repeat(np.arange(N_FIRMS), PER)
    firm_shock = r.normal(0, 1.2, N_FIRMS)[firm]     # shared within firm
    x = r.normal(0, 1, N_FIRMS * PER) + r.normal(0, 1, N_FIRMS)[firm]   # part own, part firm
    y = TRUE_B * x + firm_shock + r.normal(0, 0.6, N_FIRMS * PER)
    return pd.DataFrame({"firm": firm, "x": x, "y": y})


panel = clustered_panel(RANDOM_STATE)

**Three standard errors for one slope.** `.fit(cov_type="HC3")` fits the same regression with HC3
standard errors, as `get_robustcov_results` did in Diagnosis 1. `cov_type="cluster"` with
`cov_kwds={"groups": panel["firm"]}` clusters them by firm: it lets the errors of rows in the same
firm be correlated. `fits` is a dictionary of the three fits, and `.items()` walks through its
names and fits in pairs. `.params.iloc[1]` is the second coefficient, the slope (`.iloc` picks by
position), and `.bse` holds the standard errors.

In [ ]:
# GIVEN — run as-is
# Diagnosis 2, continued: one regression, three standard errors for its slope
base = sm.OLS(panel["y"], sm.add_constant(panel["x"]))
fits = {
    "classical": base.fit(),
    "HC3 robust": base.fit(cov_type="HC3"),
    "clustered by firm": base.fit(cov_type="cluster",
                                  cov_kwds={"groups": panel["firm"]}),
}
print(f"{'cov type':>20}{'estimate':>11}{'std err':>10}")
print("-" * 42)
for name, f in fits.items():
    print(f"{name:>20}{f.params.iloc[1]:>11.4f}{f.bse.iloc[1]:>10.4f}")

**Expected output:** the estimate is 1.4927 in all three rows; the standard errors are 0.0254
(classical), 0.0264 (HC3) and 0.0879 (clustered by firm).

**Which one actually covers?** The next cell repeats Diagnosis 1's count on 300 new panels, each
with its own seed. With a DataFrame as input, `.conf_int()` gives a table, so `.iloc[1]` picks the
slope's row. It takes a few seconds.

In [ ]:
# GIVEN — run as-is
# Diagnosis 2, continued: which interval actually covers?
TRIALS = 300
covered = {"classical": 0, "HC3": 0, "cluster": 0}     # intervals that contained TRUE_B
for i in range(TRIALS):
    sim_panel = clustered_panel(7000 + i)
    sim_model = sm.OLS(sim_panel["y"], sm.add_constant(sim_panel["x"]))
    for name, fit in [("classical", sim_model.fit()),
                      ("HC3", sim_model.fit(cov_type="HC3")),
                      ("cluster", sim_model.fit(cov_type="cluster",
                                                cov_kwds={"groups": sim_panel["firm"]}))]:
        lo, hi = fit.conf_int().iloc[1]
        covered[name] += lo <= TRUE_B <= hi

for name, count in covered.items():
    p = count / TRIALS
    print(f"{name:>10}: {p:6.1%}  (MC SE {(p*(1-p)/TRIALS)**0.5:.1%})   nominal 95%")

**Expected output:** classical 49.0% (MC SE 2.9%), HC3 50.3% (MC SE 2.9%), cluster 93.7% (MC SE
1.4%).

**The analyst's file.** Real files are messier than `panel`. The next cell gives the same 1,000
employees the columns an analyst would receive: the firm's industry (8 industries of 5 firms),
each employee's tenure in years, missing for 40 who did not report it, and a manager rating from 1
to 5, missing for 60, which the regression after it does not use. Tenure is drawn for each
employee separately, with no part shared within a firm.

**Reading the code.** `panel.copy()` is a separate copy, so `panel` itself does not change.
`% 8` is the remainder after dividing by 8, so firms 0, 8, 16, 24 and 32 share industry 0.
`.round(1)` keeps one decimal. `file_rng.choice(1000, 40, replace=False)` picks 40 different row
numbers, and `.loc[rows, "tenure"] = np.nan` empties those cells (`np.nan` is a missing value).
`file_rng.integers(1, 6, n)` draws whole numbers from 1 to 5: the upper limit is left out.
`.astype(float)` stores the ratings as decimal numbers, because a column of whole numbers cannot
hold `np.nan`. `.isna().sum()` counts the missing values in each column, and `.to_string()` prints
the counts without the summary line pandas adds below a column.

In [ ]:
# GIVEN — run as-is
# Diagnosis 2, continued: the analyst's file, the same 1,000 employees with three more columns
file_rng = np.random.default_rng(14)                 # its own seed, so the panel is untouched
staff = panel.copy()
staff["industry"] = staff["firm"] % 8
staff["tenure"] = file_rng.uniform(0, 20, len(staff)).round(1)
staff.loc[file_rng.choice(len(staff), 40, replace=False), "tenure"] = np.nan
staff["rating"] = file_rng.integers(1, 6, len(staff)).astype(float)
staff.loc[file_rng.choice(len(staff), 60, replace=False), "rating"] = np.nan
print(staff.isna().sum().to_string())

**Expected output:** 40 missing values in `tenure`, 60 in `rating`, and none elsewhere.

**Every standard error statsmodels offers, on one regression.** The analyst regresses y on x and
tenure. `.dropna(subset=[...])` keeps the rows where none of those three columns is missing,
which are the rows the regression can use. The cell fits the same model six ways and keeps each
fit's `.bse` as one column of a table: classical, the four heteroskedasticity-robust types HC0 to
HC3, and clustered by firm. `pd.DataFrame({...})` lines the six columns up by coefficient name,
and `float_format="{:.5f}".format` prints every number with five decimals.

**The small-sample corrections, as statsmodels makes them by default.** HC1 multiplies HC0's
variance by n/(n − k), where n is the number of rows used and k the number of coefficients; HC2
and HC3 divide each squared residual by (1 − h) and (1 − h)², where h is the row's leverage. The
clustered variance is multiplied by G/(G − 1) · (N − 1)/(N − K), where G is the number of clusters,
N the rows used and K the coefficients.

In [ ]:
# GIVEN — run as-is
# Diagnosis 2, continued: the analyst's regression, with six kinds of standard error
used = staff.dropna(subset=["y", "x", "tenure"])     # the rows this regression can use
model = sm.OLS(used["y"], sm.add_constant(used[["x", "tenure"]]))
ladder = pd.DataFrame({
    "classical": model.fit().bse,
    "HC0": model.fit(cov_type="HC0").bse,
    "HC1": model.fit(cov_type="HC1").bse,
    "HC2": model.fit(cov_type="HC2").bse,
    "HC3": model.fit(cov_type="HC3").bse,
    "cluster": model.fit(cov_type="cluster", cov_kwds={"groups": used["firm"]}).bse,
})
print(f"{len(used)} of {len(staff)} rows used")
print(ladder.to_string(float_format="{:.5f}".format))

**Expected output:** 960 of 1000 rows used, and this table:

| | classical | HC0 | HC1 | HC2 | HC3 | cluster |
|---|---|---|---|---|---|---|
| const | 0.07454 | 0.07763 | 0.07775 | 0.07779 | 0.07794 | 0.19509 |
| x | 0.02576 | 0.02652 | 0.02657 | 0.02660 | 0.02667 | 0.08644 |
| tenure | 0.00643 | 0.00651 | 0.00652 | 0.00652 | 0.00654 | 0.00629 |

The AI-Assisted Expansion asks you to reproduce this table with a function written by AI.

### YOUR DIAGNOSIS

1. In the coverage count, HC3 barely improves on classical and clustering restores coverage. Say what HC3 corrects for and why that is the wrong correction for this data, then what clustering assumes about the dependence structure and what would break it.
2. In the analyst's table, compare the classical and clustered columns for x and for tenure. Using how the two were generated, explain why clustering changes one so much more than the other.

*Your diagnosis here:*

---

## Diagnosis 3: Influence at Scale (20 min)

The next cell adds three rows, A, B and C, to a copy of the panel, `dirty`, and fits the slope with
and without them. For comparison, x in the panel runs from about −4.1 to 4.8, and the true line is
y = 1.5x plus each firm's shock.

**Reading the code.** `index=["A", "B", "C"]` names the three planted rows, so you can find them
later, and `pd.concat([panel, planted])` stacks the two tables. `.params["x"]` is the slope.

In [ ]:
# GIVEN — run as-is
# Diagnosis 3: three planted rows, and the slope with and without them
planted = pd.DataFrame({"firm": [0, 1, 2], "x": [8.0, 0.0, 7.0], "y": [12.0, 8.0, -4.0]},
                       index=["A", "B", "C"])
dirty = pd.concat([panel, planted])                  # 1,003 rows: the panel, then A, B and C

clean_fit = sm.OLS(panel["y"], sm.add_constant(panel["x"])).fit()
dirty_fit = sm.OLS(dirty["y"], sm.add_constant(dirty["x"])).fit()
print(f"slope without the planted rows: {clean_fit.params['x']:.4f}")
print(f"slope with them               : {dirty_fit.params['x']:.4f}")

**Expected output:** a slope of 1.4927 without the planted rows and 1.4468 with them.

**Your task.** In the empty cell below, for `dirty_fit`:
1. Build one table with a row for each row of `dirty` (`index=dirty.index`) and three columns:
   leverage, studentised residual and Cook's distance. `dirty_fit.get_influence()` computes all
   three: its `.hat_matrix_diag` is the leverage, `.resid_studentized_external` the studentised
   residual, and `.cooks_distance` a pair whose first element is Cook's distance.
2. Print the three planted rows, with `.loc[["A", "B", "C"]]`.
3. Print how many rows the lecture's two flags catch: Cook's distance above 4/n, and leverage above
   3p/n, where n is the number of rows and p = 2, the number of coefficients. `(s > c).sum()`
   counts the entries of `s` above `c`.

In [ ]:
# YOUR TASK — Diagnosis 3: leverage, studentised residual and Cook's distance for dirty_fit

### YOUR DIAGNOSIS

1. Of the three planted rows, which has the largest leverage, which the largest studentised residual, and which the largest Cook's distance? Only one of the three moved the slope from 1.4927 to 1.4468: which, and why do the other two barely move it?
2. You found them because you planted them. Write the rule you would use on data you did not build, and state its false-positive cost: how many of the rows your counts flag did nobody plant, and what would deleting every flagged row do to a sound analysis?

*Your diagnosis here:*

---

## Ship It: `src/diagnostics.py` (7 min)

Here is the finished module, reading only: run the two cells and compare the output with the
expected output below. `recommend_cov` gives back the covariance type to use and the reason,
decided from the design, not from a test on the residuals: HC3 when the rows are independent,
clustered errors when you pass the groups, and a warning below 40 clusters (Chapter 14).
`fit_as_recommended` fits OLS with that choice.

`%%writefile src/diagnostics.py` saves the cell as that file instead of running it. `groups=None`
makes `groups` optional, `-> dict` only notes the type given back, and `.nunique()` counts the
distinct groups.

In [ ]:
%%writefile src/diagnostics.py
"""Regression diagnostics that end in a recommendation (ECON 5200 Lab 14).

The design rule: a diagnostic that reports a statistic and stops leaves the reader to guess, so
recommend_cov gives back the covariance type to use and the reason for it.
"""
import pandas as pd
import statsmodels.api as sm

FEW_CLUSTERS = 40      # below about 40 clusters, cluster-robust tests reject too often (Ch 14)


def recommend_cov(groups=None) -> dict:
    """The covariance type the design calls for, chosen before looking at any residuals."""
    if groups is None:
        return {"cov_type": "HC3", "cov_kwds": {},
                "reason": "rows treated as independent: HC3 by default, whatever a test says"}
    n_groups = pd.Series(groups).nunique()
    reason = f"errors may be shared within each of {n_groups} groups: cluster on them"
    if n_groups < FEW_CLUSTERS:
        reason += f"; only {n_groups} clusters, so check with a wild cluster bootstrap"
    return {"cov_type": "cluster", "cov_kwds": {"groups": groups}, "reason": reason}


def fit_as_recommended(y, X, groups=None):
    """OLS of y on X plus an intercept, with the recommended covariance type, and the advice."""
    advice = recommend_cov(groups)
    fit = sm.OLS(y, sm.add_constant(X)).fit(cov_type=advice["cov_type"],
                                            cov_kwds=advice["cov_kwds"])
    return fit, advice

In [ ]:
# GIVEN — run as-is
# Import the module and ask it about three designs
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import diagnostics
importlib.reload(diagnostics)      # reload, so a re-run of the %%writefile cell takes effect

eight = panel[panel["firm"] < 8]
designs = [("panel, firms given", panel, panel["firm"]),
           ("panel, no groups given", panel, None),
           ("8 firms, firms given", eight, eight["firm"])]
for label, data, groups in designs:
    fit, advice = diagnostics.fit_as_recommended(data["y"], data["x"], groups)
    print(f"{label:<24}{advice['cov_type']:<9}slope SE {fit.bse['x']:.4f}   {advice['reason']}")

**Expected output:** with the firms given, the module clusters, and the slope's standard error is
0.0879, as in Diagnosis 2. With no groups given it falls back to HC3 and 0.0264, the standard error
whose interval covered about half the time: the module cannot see clustering you do not tell it
about. With 8 firms it clusters (0.2399) and adds the warning; the Extension measures what that
warning is about.

---

## AI-Assisted Expansion: a table of standard errors, your prompt, checked by you

Diagnosis 2's last cell fitted one model six times to build one table of standard errors. A
function that builds that table for any regression is code you would reuse on every project, and
quick for AI to write. It also makes every choice your prompt leaves out, and some of those choices
move a standard error by less than one percent, which is why the check prints every number. No
prompt is given: you write it.

**The target.** Given the analyst's file `staff`, the outcome `y`, the regressors `x` and `tenure`,
and the cluster variable `firm`, your function must reproduce Diagnosis 2's table: all 18 standard
errors, from const's classical **0.07454** to tenure's clustered **0.00629**.

**What the check calls.** `se_table(df, outcome, regressors, cluster)`: a DataFrame `df`, the name
of the outcome column, a list of the regressor column names, and the name of the cluster column.
It gives back a pandas DataFrame with one row per coefficient (`const` first, then the regressors
in order) and one column per type of standard error, named `classical`, `HC0`, `HC1`, `HC2`, `HC3`
and `cluster`. The check runs it on a copy of `staff` with `"y"`, `["x", "tenure"]` and `"firm"`,
then again on a copy with every column renamed, where it should give the same 18 numbers.

**What a good prompt states.** Diagnosis 2 holds every answer.
- the function's name, its arguments in order, and what each one is;
- the data: what is in `staff`, and anything about it the AI cannot see from the column names;
- the model: the outcome, the regressors, and whether there is an intercept;
- which standard errors, by name, and which column the clustered ones group by;
- the small-sample corrections, for the robust types and for the clustered one;
- what to give back: the table's rows, its columns and their names;
- what to leave out (example data, a fit of its own), and a comment on each line.

**What to do**
1. Write your round 1 prompt in the cell below and send it, exactly as written, to Claude
   ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT, in a new chat.
2. Read the reply before you run anything. Paste only the function into the round 1 cell, and run
   the check below it.
3. If a line says `NO` or `missing`, or the function stops with an error, find the choice in the
   function that caused it, state that choice in a round 2 prompt, and paste and check the new
   function in the round 2 cells. A third round goes over round 2's function. Stop when every line
   says `yes` and you can explain each line, or when you decide to stop.
4. If everything says `yes` in round 1, skip rounds 2 and 3. Instead, in Question 1, name a
   choice the AI made that your prompt never stated, and write the line you would add to your
   prompt to state it.
5. Fill in the checking log and answer the two questions. Those and your prompts are graded, not
   the code.

**Reading the check.** It prints one line per standard error: the coefficient and the type,
statsmodels' number from Diagnosis 2, the AI's, `same?`, and the % apart. Then it prints how many
of the 18 the renamed copy reproduces. Your numbers and the AI's come from the same data in the
same session, so a correct function agrees in every printed digit. There is no allowed gap: a line
a fraction of a percent apart says `NO`, and it means the function did something different, even if
that something is only rounding. The check does not say why a line differs: finding that out is
your job.

Keep the function's name and argument order. If the AI also wrote example data or fitted a model
of its own, leave that out: it can replace the lab's.

In [ ]:
# GIVEN — run as-is (you need not follow this code; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves the analyst's file, the panel and Diagnosis 2's table now, before anything is pasted.
saved_staff, saved_panel, saved_ladder = staff.copy(), panel.copy(), ladder.copy()

# The second run renames every column; a correct function gives the same 18 numbers
RENAMED = {"firm": "employer", "x": "training", "y": "salary", "industry": "sector",
           "tenure": "years", "rating": "score"}
# The names the check reads in the AI's table, in lower case: a column name -> the lab's type
TYPE_NAMES = {"classical": "classical", "nonrobust": "classical", "hc0": "HC0", "hc1": "HC1",
              "hc2": "HC2", "hc3": "HC3", "cluster": "cluster", "clustered": "cluster"}
# ... and a row name -> the lab's coefficient (training and years are the renamed copy's)
COEF_NAMES = {"const": "const", "intercept": "const", "x": "x", "tenure": "tenure",
              "training": "x", "years": "tenure"}


def lab_data_unchanged():
    """True if staff, panel and Diagnosis 2's table still hold what the lab made."""
    # try runs the lines under it; if they raise an error (say staff is gone), except runs instead
    try:
        return (staff.equals(saved_staff) and panel.equals(saved_panel)
                and ladder.equals(saved_ladder))
    except Exception:
        return False


def with_lab_names(table):
    """The table with its rows and columns renamed to the lab's names, where the check knows them."""
    return table.rename(columns=lambda c: TYPE_NAMES.get(str(c).strip().lower(), c),
                        index=lambda r: COEF_NAMES.get(str(r).strip().lower(), r))


def as_frame(result):
    """What the function gave back, as a DataFrame with the AI's own names, or None."""
    if isinstance(result, (tuple, list)):          # (table, something else)
        tables = [r for r in result if isinstance(r, (pd.DataFrame, pd.Series, dict))]
        result = tables[0] if tables else result
    if isinstance(result, dict):
        try:
            result = pd.DataFrame(result)
        except ValueError:                         # a dict of single numbers
            result = pd.Series(result)
    if isinstance(result, pd.Series):
        result = result.to_frame()
    return result if isinstance(result, pd.DataFrame) else None


def as_table(frame):
    """The AI's DataFrame with the lab's names, turned round if the types are its rows."""
    if frame is None:
        return None
    table = with_lab_names(frame)
    if not set(table.columns) & set(TYPE_NAMES.values()):   # the types are rows: turn it round
        turned = with_lab_names(frame.T)
        if set(turned.columns) & set(TYPE_NAMES.values()):
            table = turned
    return table


def show_pair(label, mine, ai):
    """One line: statsmodels' number, the AI's, whether they agree, and how far apart they are."""
    # rtol=1e-9: equal to about 9 significant digits, so only rounding inside the computer passes
    same = "yes" if np.isclose(mine, ai, rtol=1e-9, atol=0) else "NO"
    pct = 100 * (ai - mine) / abs(mine)
    # sign and 2 significant digits, so a tiny gap prints like +9.6e-06 and 1.37 as +1.4;
    # from 100% on, a whole number with its sign, like -200
    apart = f"{pct:+.2g}" if abs(pct) < 100 else f"{pct:+.0f}"
    print(f"  {label:<18}{mine:>12.5f}{ai:>10.5f}{same:>7}{apart:>11}%")


def value_in(table, coef, kind):
    """The number in the AI's table for this coefficient and type, or None."""
    if table is None or coef not in table.index or kind not in table.columns:
        return None
    try:
        return float(table.loc[coef, kind])
    except (TypeError, ValueError):
        return None


def check_se_table(fn):
    """Run the AI's se_table on the analyst's file, beside Diagnosis 2's table."""
    if not lab_data_unchanged():
        print("The pasted code changed staff, panel or Diagnosis 2's table (ladder). "
              "Paste only the function, then rerun the notebook from the top.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("se_table should be a function. Paste the function the AI wrote.")
        return
    try:
        result = fn(saved_staff.copy(), "y", ["x", "tenure"], "firm")
    except Exception as err:
        print(f"The AI's function stopped with an error. {type(err).__name__}: {err}")
        return
    frame = as_frame(result)
    if frame is None:
        print(f"se_table gave back a {type(result).__name__}. The check needs a pandas DataFrame.")
        return
    table = as_table(frame)
    print(f"the AI's table: rows {[str(r) for r in frame.index]}, "
          f"columns {[str(c) for c in frame.columns]}")
    shown = ([str(r) for r in table.index], [str(c) for c in table.columns])
    if shown != ([str(r) for r in frame.index], [str(c) for c in frame.columns]):
        print(f"read as       : rows {shown[0]}, columns {shown[1]}")
    print(f"\n  {'':<18}{'statsmodels':>12}{'AI':>10}{'same?':>7}{'% apart':>12}")
    for coef in saved_ladder.index:
        for kind in saved_ladder.columns:
            label = f"{coef}, {kind}"
            ai = value_in(table, coef, kind)
            if ai is None:
                print(f"  {label:<18}{saved_ladder.loc[coef, kind]:>12.5f}{'missing':>10}")
            else:
                show_pair(label, saved_ladder.loc[coef, kind], ai)

    renamed_note = "With every column renamed (salary, training, years, employer)"
    try:
        renamed = fn(saved_staff.rename(columns=RENAMED), "salary", ["training", "years"],
                     "employer")
    except Exception as err:
        print(f"\n{renamed_note}, the AI's function stopped with an error. "
              f"{type(err).__name__}: {err}")
    else:
        table2 = as_table(as_frame(renamed))
        agree = sum(value_in(table2, coef, kind) is not None
                    and np.isclose(value_in(table2, coef, kind), saved_ladder.loc[coef, kind],
                                   rtol=1e-9, atol=0)
                    for coef in saved_ladder.index for kind in saved_ladder.columns)
        print(f"\n{renamed_note}: {agree} of 18 the same as statsmodels   "
              f"{'yes' if agree == 18 else 'NO'}")
    print("\nIf a line says NO or missing, find out why before you trust the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the se_table function from round 1 here, then run the check below.
# Paste only the function: if the AI also wrote example data or fitted a model, leave it out.

In [ ]:
# CHECK — round 1
# To rerun this check, rerun the round 1 paste cell first: it checks whatever was pasted last
# globals() holds every name defined so far in this notebook
if "se_table" not in globals():
    print("Paste the AI's se_table function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    se_table_r1 = se_table      # kept, so the round 2 check can tell the two apart
    check_se_table(se_table_r1)

**Your prompt, round 2** (and round 3 under it, if you need one). Keep what worked in round 1,
and state what the AI had to choose on its own.

*Your prompt here:*

In [ ]:
# Paste the se_table function from round 2 here, then run the check below.
# Paste only the function: if the AI also wrote example data or fitted a model, leave it out.

In [ ]:
# CHECK — round 2 (and round 3)
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "se_table" not in globals() or se_table is globals().get("se_table_r1"):
    print("Paste the AI's round 2 se_table function in the cell above first, "
          "keeping that name, and run that cell.")
else:
    check_se_table(se_table)

### Checking log

This is graded, with your prompts and the two answers below. One row per round (a third round, if
any, goes in round 2's row): what you checked, how you checked it, and what you found. A row that
names the lines that differed, how far apart they were, and the line of the AI's code responsible
earns credit; "it worked" does not. The last row is where you stopped, and why.

| Round | What I checked | How | What I found |
|---|---|---|---|
| 1 |  |  |  |
| 2 (write "not needed" if round 1 was all yes) |  |  |  |
| Where I stopped |  |  |  |

### Questions

1. **Critique, in your own words.** Name one choice the AI made that your prompt never stated, quote the line where it makes it, and say whether you would keep it and why. (A choice that caused a `NO`, a `missing` or an error counts. If everything said `yes` in round 1, name a choice your prompt never stated and write the prompt line that would state it; then name a line that would give different numbers or break on another model or data: for example a cluster column with missing values, a regression without an intercept, or a single regressor passed as a name rather than a list.) Then quote the lines of your final function that compute the clustered standard errors and explain in 2-3 sentences, in your own words, what they do and why they give Diagnosis 2's numbers.
2. **Judgment: where you stopped.** Which round did you stop at, and why do you, or do you not, trust that function on a model and data the check has never seen? Then: when would you ask AI to write code like this, and when would you write it yourself?

*Your answers here:*

1.
2.

---

## Extension (Optional — 40 min): The Wild Cluster Bootstrap

With few clusters, cluster-robust errors over-reject. Chapter 11 (Section 11.6, "Few clusters")
named the fix, the wild cluster bootstrap, and Ship It's module warns below 40 clusters.

**The task.** In the cell below, and in new cells under it:
1. Write a generator like `clustered_panel`, with only 8 firms of 25 employees and a true slope of
   zero. Give it its own names, so Diagnosis 2's `panel` and the AI check are untouched.
2. For each of at least 500 panels, test slope = 0 at 5% three ways: with classical standard
   errors, with errors clustered by firm, and with a wild cluster bootstrap that imposes the null.
   For the clustered test, use the clustered fit's `.pvalues`, which compare the t-statistic with
   the normal distribution. For the bootstrap, fit y on the intercept alone (the null), multiply each firm's residuals by a
   Rademacher weight (+1 or −1, one per firm), rebuild y, refit, and compare your clustered
   t-statistic with the bootstrap ones. With 8 firms there are only 2⁸ = 256 sign patterns, so you
   can use all of them.
3. Report the three rejection rates, each with its Monte Carlo SE. The nominal rate is 5%.

If you skip the Extension, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — optional: the wild cluster bootstrap with 8 firms

### Interpretation Questions

1. Report the three rejection rates. Which are within two Monte Carlo SEs of 5%?
2. With 40 firms, Diagnosis 2's clustered interval covered 93.7%. At roughly what number of clusters would you switch from cluster-robust errors to the wild cluster bootstrap, and why does the number of clusters matter rather than the number of rows?

*Your answers here:*

1.

2.

---
## Digital Portfolio: your README

Copy the prompt below into Claude or ChatGPT. Replace every `[YOUR VALUE]` with the number from
your own output first. **Do NOT ask the AI to write Python code — only documentation.**

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Measured how often a nominal 95% interval contains the true slope:
  with heteroskedastic errors, [YOUR VALUE] for classical standard
  errors and [YOUR VALUE] for HC3; with errors shared within 40 firms,
  [YOUR VALUE] for HC3 and [YOUR VALUE] when clustered by firm
* Found three planted rows with leverage, studentised residuals and
  Cook's distance; one of them moved the slope from [YOUR VALUE] to
  [YOUR VALUE]
* Wrote my own prompt asking an AI for a table of standard errors, went
  [YOUR VALUE] rounds, and checked all 18 numbers against statsmodels:
  [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: OLS Diagnostics: Robust and Clustered Standard Errors
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'cutting-edge'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab14-ols-diagnostics`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab14-ols-diagnostics`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab14-ols-diagnostics`, branch `main`,
   commit message `Lab 14 submission`. This pushes the notebook, and the
   notebook's rendered output is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/diagnostics.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.